# Python bridge for 04a

## A short code refresher

Use this optional notebook alongside 04a for a closer look at the Python mechanics of PCA. Plan for about 15–20 minutes, plus a few minutes if you work through the optional reconstruction section. It reuses the small feature table from the 03a bridge and includes all the setup here.

The 04a lecture develops the principal directions and their interpretation. This companion follows five code patterns:

1. save and apply feature means and scales;
2. create and fit a PCA object;
3. distinguish its loadings from observation scores;
4. inspect component variances and retained percentages; and
5. transform one row using the same fitted model.

## Create a feature table

Rows represent four made-up records. Each has three numerical features, matching the rows-by-features arrangement used in 04a.

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA

In [ ]:
features = pd.DataFrame(
    {
        "feature_1": [10, 13, 12, 11],
        "feature_2": [50, 50, 52, 51],
        "feature_3": [2, 2, 2, 6],
    },
    index=["A", "B", "C", "D"],
)

print("feature-table shape:", features.shape)
features

The row labels identify the records. Only the three feature columns enter PCA.

## Save the means and scales

As in 03a, subtract each column's mean and divide by its sample standard deviation. pandas matches the means and scales to columns by name.

In [ ]:
feature_means = features.mean()
feature_sds = features.std(ddof=1)
standardized = features.sub(feature_means).div(feature_sds)

standardized.round(3)

Each standardized column has mean approximately zero and sample variance one. `ddof=1` uses the sample convention from 04a. All three columns vary, so their standard deviations are nonzero. PCA centers the data passed to it but does not divide by these scales automatically.

## Create and fit the PCA object

`PCA(...)` creates a configurable object. Here `n_components=2` asks it to retain two component directions, and `svd_solver="full"` uses the same fitting method as 04a. Calling `.fit(standardized)` learns from all four rows and stores the result on `model`.

In [ ]:
model = PCA(n_components=2, svd_solver="full")
model.fit(standardized)

print("input features:", model.n_features_in_)
print("retained components:", model.n_components_)
print("fitted input means:", model.mean_.round(10))

The trailing underscore marks information learned during fitting. `model.mean_` contains the means of the standardized inputs, which are approximately zero. `feature_means` still contains the original means. Keep both roles clear: `.fit()` returns the fitted object, while `.transform()` will return scores.

## Inspect the loadings

`model.components_` has one row per retained component and one column per original feature. Following 04a, we call these direction coefficients **loadings**. The transpose, `.T`, lets us display a component's loadings down one column.

In [ ]:
component_names = [f"PC{i + 1}" for i in range(model.n_components_)]
loadings = pd.DataFrame(
    model.components_.T, index=features.columns, columns=component_names
)

print("stored coefficient-array shape:", model.components_.shape)
loadings.round(3)

The stored array has shape `(2, 3)`, and the transposed table has shape `(3, 2)`. For example, `loadings.loc["feature_1", "PC1"]` is the coefficient used for feature 1 in every record's PC1 score. The name-building expression creates one label per fitted component, so it also works when the component count changes.

## Calculate observation scores

`.transform()` applies the fitted directions to each row. A **score** is one record's coordinate along a component. The returned array preserves the input row order, so attach the original record labels when displaying it.

In [ ]:
scores = model.transform(standardized)
score_frame = pd.DataFrame(scores, index=features.index, columns=component_names)

print("score-array shape:", scores.shape)
score_frame.round(3)

The `(4, 2)` table has four records and two component scores per record. All records use the same loadings, but their measurements give them different scores.

We can check one score directly. Subtract the means stored by PCA, multiply the input values by the PC1 loadings, and add those contributions.

In [ ]:
centered_a = standardized.loc["A"] - model.mean_
pc1_contributions = centered_a * loadings["PC1"]

print("PC1 contributions for A:")
print(pc1_contributions.round(3))
print("sum of contributions:", round(pc1_contributions.sum(), 3))
print("PCA's score for A:", round(score_frame.loc["A", "PC1"], 3))

The sum and PCA's score agree, about 0.257 in the displayed orientation. Use the full stored coefficients for calculations. `.round(3)` only shortens what we display.

## Inspect the retained variance

The PCA object also stores the sample variance of each component's scores and its share of the total input variance. Multiplication by 100 expresses the shares as percentages. `.cumsum()` adds them in component order.

In [ ]:
variance_summary = pd.DataFrame(
    {
        "Component variance": model.explained_variance_,
        "Variance retained (%)": 100 * model.explained_variance_ratio_,
        "Cumulative retained (%)": 100 * model.explained_variance_ratio_.cumsum(),
    },
    index=component_names,
)

variance_summary.round(3)

PC1 retains about 42.09% of the standardized variance. PC2 adds about 37.35%, bringing the two-component total to 79.44%. The component variances are the eigenvalues discussed in 04a. Each describes spread across records, whereas an individual score describes one record's position. These retained percentages refer to all three input features, even though we requested only two components.

## Transform one row with the fitted model

Double brackets preserve the two-dimensional input expected by scikit-learn. Standardize the selected record with the saved original means and scales, then use the model already fitted to the whole table.

In [ ]:
one_record = features.loc[["A"]]
one_standardized = one_record.sub(feature_means).div(feature_sds)
one_scores = model.transform(one_standardized)

print("one-row input shape:", one_standardized.shape)
print("one-row score shape:", one_scores.shape)
pd.DataFrame(one_scores, index=one_record.index, columns=component_names).round(3)

The input has shape `(1, 3)` and the result has shape `(1, 2)`. A's scores match its row in the earlier score table. For another observation, reuse the same feature order, means, scales, and fitted model. Recalculating them from the selected row would change the transformation.

## Try one small modification

Change `n_components=2` to `n_components=1`, then rerun the fitting cell and all later code cells. The score table should have one column, and the cumulative retained percentage should end at about 42.09%. The original feature table still has three columns. Restore two components before continuing if you want to compare with the original displays.

## Optional: recover approximate measurements

`inverse_transform(scores)` combines the retained scores with the fitted directions and restores the model's input means. Because we fitted standardized data, its output is also standardized. Multiply by the saved original standard deviations and add the original means to recover the measurement scales.

In [ ]:
rebuilt_standardized = pd.DataFrame(
    model.inverse_transform(scores), index=features.index, columns=features.columns
)
rebuilt = rebuilt_standardized.mul(feature_sds).add(feature_means)

pd.DataFrame(
    {"Original A": features.loc["A"], "Reconstructed A": rebuilt.loc["A"]}
).round(3)

With fewer components, some variation is discarded and the recovered values can differ from the originals. The score table loses columns, but reconstruction returns one value for every original feature. The [Course Addenda and Errata page](https://auburn.instructure.com/courses/1747056/pages/course-addenda-and-errata) explains the calculation further under “04A: Reconstructing Measurements from PCA Scores.”

## Ready for 04a

You are ready to return to 04a when you can recognize these patterns:

```text
features.sub(means).div(sds)             # apply the saved feature scales
model = PCA(n_components=2)              # create a configurable object
model.fit(standardized)                  # learn directions from the input rows
model.components_                       # inspect the fitted direction coefficients
model.transform(standardized)            # calculate observation scores
model.explained_variance_ratio_.cumsum() # accumulate retained variance shares
```

Check which object each step returns and which labels belong on it. Loadings connect input features to components. Scores connect records to components. Keep the saved preprocessing values with the fitted model so another row receives the same transformation.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).